# D242 — Handling JSON Data in Apache Hive

This lab demonstrates how Hive reads, catalogs, queries, and converts JSON data. It follows the D241 workflow: create self-contained files with `tee`, upload them to HDFS, register tables, and query them through HiveServer2.

**Environment:** Hive 4.0.1, Hadoop 3.3.6, HDFS, YARN/MapReduce, and Beeline.

> Every input record in this notebook is one complete JSON object on one physical line. This format is commonly called **JSON Lines**, **newline-delimited JSON**, or **NDJSON**. The files do not contain a header row.

## Learning path

1. Understand JSON handling in Hive
2. Generate and upload JSON Lines data
3. Read raw JSON text with JSON functions
4. Create an external table with `JsonSerDe`
5. Query nested structures, arrays, and maps
6. Create and load a managed JSON table
7. Use CTAS to convert JSON into an ORC analytics table
8. Validate malformed JSON before production use

## 0. Service spot-check

In [ ]:
%%bash
jps
echo '--- Hive listeners ---'
ss -lnt | grep -E ':(9083|10000|10002)\b' || true
echo '--- HDFS health ---'
hdfs dfsadmin -report | grep -E 'Live datanodes|Name:'

Expected: one live DataNode, the Hadoop daemons, Hive metastore, and HiveServer2 on port `10000`.

# 1. How Hive handles JSON

JSON is a text serialization format, while a Hive table has typed columns. Hive needs a **SerDe**—serializer/deserializer—to translate between JSON objects and Hive rows. This lesson uses Hive's HCatalog JSON SerDe:

```text
org.apache.hive.hcatalog.data.JsonSerDe
```

On reads, the deserializer maps JSON keys to Hive columns and converts values to the declared Hive types. Complex JSON values can map to `STRUCT`, `ARRAY`, and `MAP`.

Hive can also store each JSON object as a plain `STRING` and extract fields with functions such as `get_json_object` and `json_tuple`. This raw-text approach is flexible during discovery; a SerDe table is cleaner for repeated typed queries.

### JSON-to-Hive type mapping

| JSON value | Typical Hive type | Example |
|---|---|---|
| integer | `INT` or `BIGINT` | `1001` |
| decimal number | `DOUBLE` or `DECIMAL(p,s)` | `1250.50` |
| string | `STRING`, `DATE`, or `TIMESTAMP` | `"Bengaluru"` |
| boolean | `BOOLEAN` | `true` |
| object | `STRUCT<...>` or `MAP<STRING,...>` | `{"city":"Chennai"}` |
| array | `ARRAY<type>` | `["new","mobile"]` |
| null | SQL `NULL` | `null` |

The declared schema must be compatible with the input. JSON keys are not a substitute for a deliberate table schema.

# 2. Generate JSON Lines input

The sample contains nested customer data, an array of items, an array of tags, and a string-to-string metadata map. Each order remains on a single line so Hive can treat one line as one record.

In [ ]:
%%bash
tee /tmp/d242_orders.json > /dev/null <<'EOF'
{"order_id":2001,"order_ts":"2026-08-18 09:15:00","customer":{"customer_id":501,"name":"Asha","city":"Bengaluru"},"items":[{"product":"Laptop Bag","quantity":1,"unit_price":1800.00},{"product":"Mouse","quantity":2,"unit_price":850.00}],"tags":["online","priority"],"attributes":{"channel":"web","payment":"upi"},"delivered":true}
{"order_id":2002,"order_ts":"2026-08-18 10:30:00","customer":{"customer_id":502,"name":"Ravi","city":"Chennai"},"items":[{"product":"Chair","quantity":1,"unit_price":7500.00}],"tags":["store"],"attributes":{"channel":"store","payment":"card"},"delivered":false}
{"order_id":2003,"order_ts":"2026-08-19 11:45:00","customer":{"customer_id":503,"name":"Meera","city":"Hyderabad"},"items":[{"product":"Notebook","quantity":5,"unit_price":120.00},{"product":"Pen Set","quantity":2,"unit_price":250.00}],"tags":["online","new-customer"],"attributes":{"channel":"mobile","payment":"wallet"},"delivered":null}
{"order_id":2004,"order_ts":"2026-08-19 14:20:00","customer":{"customer_id":504,"name":"Arun","city":"Bengaluru"},"items":[{"product":"Desk","quantity":1,"unit_price":12000.00}],"tags":[],"attributes":{"channel":"web","payment":"netbanking"},"delivered":false}
EOF
echo 'Record count:'
wc -l /tmp/d242_orders.json
echo 'JSON Lines input:'
cat /tmp/d242_orders.json

Validate the local file before uploading it. Python's standard JSON parser is used only as a validator here; Hive will read the same file from HDFS.

In [ ]:
%%bash
python3 - <<'PY'
import json
from pathlib import Path
path = Path('/tmp/d242_orders.json')
for number, line in enumerate(path.read_text().splitlines(), 1):
    json.loads(line)
print(f'Validated {number} JSON records in {path}')
PY

Upload the file into a lab-specific external-data directory. The exact cleanup makes this cell repeatable and prevents duplicate input files.

In [ ]:
%%bash
hdfs dfs -rm -r -f /user/hive/external/d242_json_orders
hdfs dfs -mkdir -p /user/hive/external/d242_json_orders
hdfs dfs -put /tmp/d242_orders.json /user/hive/external/d242_json_orders/
hdfs dfs -ls /user/hive/external/d242_json_orders
hdfs dfs -cat /user/hive/external/d242_json_orders/d242_orders.json

Create a separate database for the lesson.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/default' -n "$USER" --silent=true -e "
CREATE DATABASE IF NOT EXISTS hive_json
COMMENT 'JSON examples for the D242 lab';
DESCRIBE DATABASE EXTENDED hive_json;
"

# 3. Raw JSON text and JSON functions

A one-column text table is useful when the schema is unknown, changing, or only a few paths are needed. Since Hive's default text delimiter is Control-A rather than a printable JSON character, each complete JSON line is read into the single `json_line` column.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_json' -n "$USER" --silent=true -e "
DROP TABLE IF EXISTS orders_json_raw;
CREATE EXTERNAL TABLE orders_json_raw (json_line STRING)
STORED AS TEXTFILE
LOCATION '/user/hive/external/d242_json_orders';
SELECT json_line FROM orders_json_raw LIMIT 2;
"

`get_json_object` extracts a value using a JSONPath-like expression. Its result is a string, so cast numeric values before arithmetic.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_json' -n "$USER" --silent=true -e "
SELECT
  CAST(get_json_object(json_line, '\$.order_id') AS INT) AS order_id,
  get_json_object(json_line, '\$.customer.name') AS customer_name,
  get_json_object(json_line, '\$.customer.city') AS city,
  get_json_object(json_line, '\$.attributes.channel') AS channel
FROM orders_json_raw
ORDER BY order_id;
"

`json_tuple` extracts several top-level keys in one operation. It is commonly used with `LATERAL VIEW`. Nested paths are easier with `get_json_object` or a typed SerDe table.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_json' -n "$USER" --silent=true -e "
SELECT jt.order_id, jt.order_ts, jt.delivered
FROM orders_json_raw r
LATERAL VIEW json_tuple(r.json_line, 'order_id', 'order_ts', 'delivered') jt
AS order_id, order_ts, delivered
ORDER BY jt.order_id;
"

# 4. External typed JSON table with JsonSerDe

The external table points to the same HDFS directory but exposes typed columns. The nested `customer` object becomes a `STRUCT`; `items` becomes an array of structs; `tags` becomes an array of strings; and `attributes` becomes a string map.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_json' -n "$USER" --silent=true -e "
DROP TABLE IF EXISTS orders_json_external;
CREATE EXTERNAL TABLE orders_json_external (
  order_id INT,
  order_ts TIMESTAMP,
  customer STRUCT<customer_id:INT,name:STRING,city:STRING>,
  items ARRAY<STRUCT<product:STRING,quantity:INT,unit_price:DECIMAL(10,2)>>,
  tags ARRAY<STRING>,
  attributes MAP<STRING,STRING>,
  delivered BOOLEAN
)
ROW FORMAT SERDE 'org.apache.hive.hcatalog.data.JsonSerDe'
STORED AS TEXTFILE
LOCATION '/user/hive/external/d242_json_orders';
DESCRIBE FORMATTED orders_json_external;
"

Look for `EXTERNAL_TABLE`, the JSON SerDe library, and `/user/hive/external/d242_json_orders` in the formatted description.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_json' -n "$USER" --silent=true -e "
SELECT
  order_id,
  order_ts,
  customer.name AS customer_name,
  customer.city AS city,
  attributes['channel'] AS channel,
  tags,
  delivered
FROM orders_json_external
ORDER BY order_id;
"

A missing JSON key or explicit JSON `null` generally appears as SQL `NULL` when the value can be deserialized. Use `IS NULL`, not `= NULL`, to test it.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_json' -n "$USER" --silent=true -e "
SELECT order_id, customer.name, delivered
FROM orders_json_external
WHERE delivered IS NULL;
"

# 5. Expand JSON arrays with `LATERAL VIEW explode`

`explode(items)` emits one row per array element. Dot notation then reads fields from each item struct. This converts nested order data into an item-level relational result.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_json' -n "$USER" --silent=true -e "
SELECT
  o.order_id,
  o.customer.city AS city,
  item.product,
  item.quantity,
  item.unit_price,
  item.quantity * item.unit_price AS line_total
FROM orders_json_external o
LATERAL VIEW explode(o.items) expanded AS item
ORDER BY o.order_id, item.product;
"

Aggregate the exploded rows to calculate each order total.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_json' -n "$USER" --silent=true -e "
SELECT
  o.order_id,
  o.customer.name AS customer_name,
  SUM(item.quantity * item.unit_price) AS order_total
FROM orders_json_external o
LATERAL VIEW explode(o.items) expanded AS item
GROUP BY o.order_id, o.customer.name
ORDER BY o.order_id;
"

Expected totals: order 2001 = 3500, order 2002 = 7500, order 2003 = 1100, and order 2004 = 12000.

# 6. Managed JSON table

A normal `CREATE TABLE` creates a managed table. The table below uses the same JSON SerDe, but Hive owns its default warehouse location. We load a separate local JSON Lines file into it.

In [ ]:
%%bash
tee /tmp/d242_events.json > /dev/null <<'EOF'
{"event_id":9001,"event_type":"page_view","user_id":701,"event_ts":"2026-08-18 08:00:00","properties":{"page":"home","device":"mobile"}}
{"event_id":9002,"event_type":"add_to_cart","user_id":701,"event_ts":"2026-08-18 08:05:00","properties":{"product":"Mouse","device":"mobile"}}
{"event_id":9003,"event_type":"purchase","user_id":702,"event_ts":"2026-08-18 08:20:00","properties":{"order_id":"2002","device":"desktop"}}
EOF
cat /tmp/d242_events.json

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_json' -n "$USER" --silent=true -e "
DROP TABLE IF EXISTS events_json_managed;
CREATE TABLE events_json_managed (
  event_id INT,
  event_type STRING,
  user_id INT,
  event_ts TIMESTAMP,
  properties MAP<STRING,STRING>
)
ROW FORMAT SERDE 'org.apache.hive.hcatalog.data.JsonSerDe'
STORED AS TEXTFILE;
LOAD DATA LOCAL INPATH '/tmp/d242_events.json'
OVERWRITE INTO TABLE events_json_managed;
SELECT event_id, event_type, user_id, properties['device'] AS device
FROM events_json_managed
ORDER BY event_id;
DESCRIBE FORMATTED events_json_managed;
"

Look for `MANAGED_TABLE` and a location under `/user/hive/warehouse/hive_json.db/`. Dropping this managed table normally removes both its catalog entry and its warehouse data.

# 7. CTAS: convert JSON into ORC

JSON is convenient for ingestion but expensive to parse repeatedly. `CREATE TABLE AS SELECT` can flatten the nested data, apply types, calculate metrics, and write a managed table in ORC columnar format. The new table is a snapshot; it will not automatically receive later JSON records.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_json' -n "$USER" --silent=true -e "
DROP TABLE IF EXISTS order_items_orc;
CREATE TABLE order_items_orc
STORED AS ORC
AS
SELECT
  o.order_id,
  o.order_ts,
  o.customer.customer_id AS customer_id,
  o.customer.name AS customer_name,
  o.customer.city AS city,
  item.product,
  item.quantity,
  item.unit_price,
  item.quantity * item.unit_price AS line_total,
  o.attributes['channel'] AS channel
FROM orders_json_external o
LATERAL VIEW explode(o.items) expanded AS item;
SELECT * FROM order_items_orc ORDER BY order_id, product;
"

Inspect the CTAS result and compare it with the external JSON table.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_json' -n "$USER" --silent=true -e "
DESCRIBE FORMATTED order_items_orc;
SELECT city, SUM(line_total) AS city_total
FROM order_items_orc
GROUP BY city
ORDER BY city;
"

The CTAS table should be `MANAGED_TABLE` with an ORC input format. Expected city totals are Bengaluru 15500, Chennai 7500, and Hyderabad 1100.

# 8. Malformed JSON and schema mismatches

Production JSON can fail because of truncated lines, pretty-printed multi-line objects, inconsistent types, unexpected field names, or incompatible nested structures. Depending on the SerDe and settings, bad records may fail the query or produce nulls. Do not silently assume every null is a genuine source null.

A practical ingestion pattern is:

1. land immutable JSON Lines files in an external HDFS location;
2. validate line counts and JSON syntax;
3. query through a raw-string table when investigating new feeds;
4. apply a typed external SerDe table after understanding the schema;
5. use CTAS or `INSERT ... SELECT` to create optimized ORC/Parquet tables;
6. retain raw files so rejected or newly added fields can be reprocessed.

### A lightweight HDFS validation check

Stream the HDFS file through Python and report invalid line numbers without changing the source data.

In [ ]:
%%bash
hdfs dfs -cat /user/hive/external/d242_json_orders/*.json | python3 -c "
import json, sys
bad = []
count = 0
for count, line in enumerate(sys.stdin, 1):
    try:
        json.loads(line)
    except json.JSONDecodeError as exc:
        bad.append((count, str(exc)))
print(f'records={count}, invalid={len(bad)}')
for row in bad:
    print(row)
sys.exit(1 if bad else 0)
"

# 9. Catalog inventory and comparison

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_json' -n "$USER" --silent=true -e "
SHOW TABLES;
"

| Object | Hive representation | Data ownership | Best use |
|---|---|---|---|
| `orders_json_raw` | One raw `STRING` per JSON line | External | Discovery and path extraction |
| `orders_json_external` | Typed columns via `JsonSerDe` | External | Repeated queries over landed JSON |
| `events_json_managed` | Typed columns via `JsonSerDe` | Hive-managed | Small Hive-owned JSON dataset |
| `order_items_orc` | Flattened typed ORC columns | Hive-managed | Faster analytical queries |

JSON preserves flexible nested source records; ORC provides an efficient, strongly typed analytical representation. A common pipeline uses both.

## Optional cleanup

Keep these objects for later lessons, or remove only the D242 catalog objects:

```sql
USE hive_json;
DROP TABLE IF EXISTS order_items_orc;
DROP TABLE IF EXISTS events_json_managed;
DROP TABLE IF EXISTS orders_json_external;
DROP TABLE IF EXISTS orders_json_raw;
```

The external source remains at `/user/hive/external/d242_json_orders` after the external tables are dropped. Remove that exact HDFS directory only when the source data is no longer needed.